In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer
import tensorflow as tf
from transformers import TFDistilBertForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [2]:
df = pd.read_csv('bitcoin_sentiments_21_24.csv')
df.columns = ['date', 'text', 'sentiment_score']

In [4]:
def create_label(score):
    if score < -0.2:
        return 0
    elif score > 0.2:
        return 2
    else:
        return 1

In [5]:
df['label'] = df['sentiment_score'].apply(create_label)

In [7]:
texts = df['text'].values
labels = df['label'].values

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, 
    test_size=0.2, 
    random_state=42
)

print(f"Training Data: {len(train_texts)}")
print(f"Test Data: {len(val_texts)}")

Training Data: 9036
Test Data: 2259


In [9]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)

C:\Users\vedat\Desktop\finsmart\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vedat\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [12]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
))

BATCH_SIZE = 16

train_dataset = train_dataset.shuffle(1000).batch(BATCH_SIZE)
val_dataset = val_dataset.batch(BATCH_SIZE)

In [17]:
# ÖNEMLİ: Standart tensorflow yerine, uyumlu olan 'tf_keras'ı kullanıyoruz
import tf_keras
from transformers import TFDistilBertForSequenceClassification

# 1. Modeli İndir
model_bert = TFDistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=3
)

# 2. Modeli Derle
# Optimizer ve Loss'u direkt tf_keras içinden çağırıyoruz ki modelle uyumlu olsun
optimizer = tf_keras.optimizers.Adam(learning_rate=5e-5)
loss = tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model_bert.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

# 3. Eğitimi Başlat
print("BERT Eğitimi Başlıyor... (Bu işlem biraz sürebilir, sabırlı ol ☕)")

history_bert = model_bert.fit(
    train_dataset, 
    validation_data=val_dataset, 
    epochs=2
)

# 4. Modeli Kaydet
save_directory = "./finsmart_bert_sentiment"
model_bert.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print("Eğitim bitti ve model kaydedildi! 🚀")

TypeError: 'builtins.safe_open' object is not iterable